# ⚠️ 공유용 사본 안내 (팀 저장소 버전)

이 노트북은 권병학 개인 workspace(`workspace_gwonbyeonghag/notebooks/08_duplicate_group_weight_experiment.ipynb`)에서
**실제로 실행된 결과를 그대로 복사**한 것이다. 아래 실행 결과(표·수치·판정)는 전부 그
원본 실행에서 나온 것이며 다시 계산하지 않았다 — 이 사본에서 코드를 다시 돌리지 않았다.

**중요 — 재현 관련 주의사항**

- `WORKSPACE_ROOT`/`ONCO_AI_ROOT`/`CONFIG_PATH` 등 경로 관련 셀은 원본 개인 폴더
  구조(`workspace_gwonbyeonghag/`가 `onco-ai/`와 형제 디렉터리)를 전제로 하므로,
  이 저장소에서 그대로 재실행되지 않는다. **이 사본의 목적은 즉시 실행되는
  스크립트가 아니라, 실험 로직과 실제 관측된 결과를 정확한 코드와 함께 남기는
  것이다.**
- 원본 데이터(`train.csv`)와 4,411열 feature parquet, OOF/metrics 산출물은 이
  저장소에 커밋돼 있지 않다(대회 규정·저장소 용량 정책). 재현하려면 사용자가
  로컬에 원본 데이터를 준비하고 경로를 직접 설정해야 한다 — 자세한 절차는
  `docs/experiments/duplicate_group_weight_experiment_20260803.md`의 "재현 방법"을
  참고한다.
- 이 노트북이 참조하는 가중치 계산 로직(`duplicate_group_weight.py`, 개인
  workspace 전용 모듈)과 수학적으로 동일한 구현이 이번 PR로 팀 저장소에
  `src/cancer_hack/sample_weights.py`로 이식됐다 — `A_none`/`B_balanced`는
  기존 canonical `cancer_hack.models_gbdt.resolve_sample_weight`를 재사용하고,
  `C_balanced_profile`도 그 canonical 구현(`resolve_sample_weight("balanced+group", ...)`,
  PR #17 `f4rw`와 동일 정의)을 그대로 쓴다. `D_balanced_same_label_naive`/
  `E_same_label_rebalanced`만 이번 PR에서 새로 추가한 것이다. 새 실험을 만들
  때는 이 노트북을 복사하지 말고 `src/cancer_hack/sample_weights.py` +
  `tests/test_sample_weights.py`를 재사용하길 권장한다.
- **Group5 fold는 원본 `train.csv`의 4,384개 변이 문자열 열로 만든
  `profile_hash` 기준이다** — enc3 인코딩값이나 4,411 최종 피처로 다시 해시한
  것이 아니다(3~7절에서 자세히 설명).
- **DACON 제출은 수행하지 않았다.** 이 실험은 test 데이터를 로드하지 않고
  OOF 비교만 한다.
- **결론**: `B_balanced`(클래스 균형 가중치만) 채택. `C_balanced_profile`,
  `D_balanced_same_label_naive`, `E_same_label_rebalanced`(중복 그룹 가중치)는
  전부 기각 — B 대비 개선폭이 사전 기준(0.005) 미달이었다(11절 결과표 참고).

실행 환경: onco-ai git HEAD `26f8cc571604dde387c24c2d8a208ef6358eb59f`(PR #17 머지
직전) 기준. 이후 PR #17이 머지돼 develop HEAD가 바뀌었으나, 이 실행 자체의
숫자·판정에는 영향이 없다(권병학 개인 파이프라인은 PR #17의 학습 드라이버를
쓰지 않았다).


# 중복 그룹 가중치 실험 — 권병학 개인 실험

`06_xgb_enc3_sample27.ipynb`와 완전히 같은 피처(4,411열)·모델(XGBoost)·하이퍼파라미터 위에서
**가중치만 바꾸는 통제 실험**이다. 팀 저장소(`onco-ai`)는 읽기 전용으로만 쓰고, 이 노트북과
그 산출물은 전부 `workspace_gwonbyeonghag/` 안에만 만든다. DACON 제출·PR·git 조작은
이번 실험 범위가 아니다.

## 1. 이번 실험에서 답하려는 질문

1. 동일하거나 반복되는 변이 프로필이 학습에 과도한 영향을 주는가?
2. 중복 그룹 가중치가 OOF Macro F1과 fold 안정성에 실제로 도움이 되는가?
3. `profile_hash`만으로 그룹핑하는 것(PR #17 `f4rw` 방식)과 `(SUBCLASS, profile_hash)`로
   그룹핑하는 것(Notion 원안)이 실제로 다른 결과를 내는가?
4. Notion 원안의 단순곱(`class_weight * duplicate_weight`)이 클래스별 총가중치 균형을
   깨뜨린다는 우려가 실제 성능에도 영향을 주는가?
5. 그 문제를 고치는 교정식(E)이 문제를 해결하면서 성능도 유지하는가?
6. 결론적으로 개인 파이프라인에 어떤 가중치 방식을 채택해야 하는가(또는 채택하지 않아야
   하는가)?

민재님의 EXP_015/f4r 결과는 참고 자료일 뿐이다 — 피처·fold·산출물 조건이 완전히 다르므로
이 실험의 기준선으로 재사용하지 않는다. 이 노트북 안에서 A(가중치 없음)와 B(클래스 균형만)를
직접 만들어 기준선으로 쓴다.

## 2. 중복 프로필이 왜 문제가 되는가

같은 변이 벡터(4,384개 유전자 열이 완전히 같은 값)를 가진 행이 여러 개 있으면, 모델은
사실상 그 표본을 여러 번 학습하는 셈이다. 두 가지 서로 다른 위험이 섞여 있다.

- **CV 누수**: 그 중복 행들이 train/validation fold에 걸쳐 나뉘면, validation 점수가
  "한 번도 못 본 데이터"가 아니라 "거의 똑같이 생긴 데이터를 본 적 있는 상태"에서 나온
  것이라 낙관적으로 부풀려진다. 이건 **Group CV로 원천 차단**한다(7절).
- **가중치 쏠림**: 누수를 fold 분할로 이미 막았다고 해도, 같은 프로필이 30번 반복되면
  그 프로필의 정보(어떤 유전자 조합 → 어떤 라벨)가 학습 손실에 30배로 반영된다. 이게
  실제로 문제가 되는지, 된다면 어떤 정의(profile-only vs same-label)로 완화해야 하는지가
  이 실험의 핵심 질문이다.

이 노트북은 두 번째 위험(가중치 쏠림)만 통제 실험으로 잰다 — 첫 번째 위험(fold 누수)은
Group5 CV를 쓰는 것으로 다섯 실험 전부에서 동일하게 이미 차단된 상태에서 시작한다.

## 3. profile-only 그룹핑과 (SUBCLASS, profile_hash) 그룹핑의 차이

- **profile-only**(`profile_hash` 단독): "이 정확한 변이 패턴이 데이터셋에 몇 번 나오는가"만
  센다. 라벨이 다른 행도 같은 그룹으로 묶인다 — 예를 들어 변이가 전혀 없는(all-WT) 행은
  train 전체에서 94개가 있는데, 이 94개는 15개 서로 다른 SUBCLASS에 걸쳐 있다(팀 EDA
  문서 확인 사항). profile-only로 그룹핑하면 이 94개 전부가 "같은 중복"으로 취급돼
  똑같이 짓눌린다.
- **(SUBCLASS, profile_hash) 조인 (same-label)**: "같은 X, 같은 y가 몇 번 반복되는가"만
  센다. 위 94개 all-WT 행은 라벨별로 재분류돼 각 라벨 안에서만 중복으로 취급된다 —
  예를 들어 그중 THYM 라벨이 30개면 그 30개끼리만 하나의 그룹이고, 다른 라벨의 all-WT
  행은 별개 그룹이다.

같은 X에 다른 y가 붙는 행은 모델에게 "이 프로필은 여러 암종에서 나타날 수 있다"는
유익한 애매성 정보다. profile-only 그룹핑은 이 정보까지 중복으로 취급해 억누르지만,
same-label 그룹핑은 보존한다. 이 차이가 실제 성능에 영향을 주는지가 9~13절에서
직접 확인할 대상이다.

## 4. 기존 PR #17 방식과 Notion 원안의 차이 (읽기 전용 확인 사항)

`onco-ai` 저장소 PR #17(`feat/중복_처리_전략`, 미머지)의 `group_size_inverse_weight()`는
**profile_hash 단독** 그룹핑만 지원한다(`SUBCLASS`를 전혀 받지 않는다) — 위 3절의
profile-only 방식과 정의가 같다. 이 함수는 `f4rw`/`f4rws` 설정으로 실험됐지만(PR 본문
자체 보고), 어떤 팀 Drive 산출물에서도 실행 로그가 발견되지 않았다 — 코드는 있지만
독립적으로 검증된 적이 없다.

Notion "중복 그룹 가중치 실험" 페이지는 `(SUBCLASS, profile_hash)` 조인 그룹핑(3절의
same-label 방식)을 제안하며, `class_weight * duplicate_weight` 단순곱을 쓴다. 이 페이지
자체가 "실제 환자 빈도를 약화할 수 있으므로 별도 실험으로 비교해야 한다"고 명시한다.

이 실험은 두 정의(profile-only=C, same-label=D)를 **동일한 Group5 fold·동일한 모델** 위에서
나란히 비교해 이 차이가 실제로 유의미한지 확인한다. PR #17 코드는 참고만 하고 재사용하지
않는다(이 노트북의 `C_balanced_profile`은 같은 수학적 정의를 독립적으로 구현한 것이다).

## 5. 다섯 가지 가중치 수식

fold의 train 부분만 $N$행, $K$개 클래스가 있다고 하자.

- **A. 가중치 없음**: `sample_weight = None`
- **B. 클래스 균형만**: `compute_sample_weight(class_weight="balanced", y=y_train_fold)` — sklearn
  정의상 평균 1.
- **C. 클래스 균형 × profile 중복 역수** (profile-only, PR #17 `f4rw` 대응):
  $duplicate\_weight_i = 1 / |\{j : profile\_hash_j = profile\_hash_i\}|$ (train fold 안에서),
  $raw\_weight_i = class\_weight_i \times duplicate\_weight_i$, 이후 평균 1로 정규화.
- **D. 클래스 균형 × same-label 중복 역수** (Notion 원안, 단순곱):
  $duplicate\_weight_i = 1 / |\{j : (SUBCLASS_j, profile\_hash_j) = (SUBCLASS_i, profile\_hash_i)\}|$,
  $raw\_weight_i = class\_weight_i \times duplicate\_weight_i$, 이후 평균 1로 정규화.
  **이 방식은 클래스별 총weight 균형이 깨질 수 있다 — 9절에서 실측한다.**
- **E. 클래스 총가중치 보존 교정식**: 클래스 $c$ 안의 고유 $(SUBCLASS, profile\_hash)$
  그룹 수를 $G_c$, 행 $i$가 속한 그룹의 크기를 $n_{cg}$라 하면

  $$weight_i = \frac{N/K}{G_c \cdot n_{cg}}$$

  이 식은 정규화 없이도 이미 평균 1이고(수식 자체가 보장), 클래스별 총weight가 정확히
  $N/K$로 균등하다 — D의 결함을 고치면서 same-label 정보는 그대로 보존한다.

## 6. 데이터 및 피처 계약 확인

**이 셀이 하는 일**: `06_xgb_enc3_sample27.ipynb`가 쓰던 것과 완전히 같은 입력
(`train_enc3_sample27_inputs.parquet`, `feature_manifest_enc3_sample27.yaml`)을 읽고, 행·열
수·컬럼 구성이 기대와 같은지 확인한다.
**왜 필요한지**: 이 실험이 "가중치만 바꾼 통제 실험"이 되려면 피처가 06과 100% 같아야 한다.
**입력**: parquet 2개, manifest 1개, `onco-ai` git 상태.
**출력**: `train_df`, `manifest`, 컬럼 리스트, git HEAD 문자열.
**누수 방지 조건**: 이 셀 자체는 fold를 아직 안 나누므로 누수 위험 없음 — 계약 검증만 한다.
**확인해야 할 부분**: `assert` 실패 없이 끝까지 실행되는지, `git HEAD`가 읽기 전용으로
확인만 되고(수정 없이) 출력되는지.

In [1]:
import sys
import json
import time
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy.sparse as sp
import xgboost as xgb
import yaml
from sklearn.metrics import f1_score, confusion_matrix

WORKSPACE_ROOT = Path("..").resolve()
PROJECT_ROOT = WORKSPACE_ROOT.parent
ONCO_AI_ROOT = PROJECT_ROOT / "onco-ai"
CONFIG_PATH = WORKSPACE_ROOT / "configs" / "xgb_enc3_sample27_duplicate_weight.yaml"

sys.path.insert(0, str(ONCO_AI_ROOT / "src"))
sys.path.insert(0, str(WORKSPACE_ROOT / "lib"))
from cancer_hack.features_basic import RatioTransformFeatures
from cancer_hack.validation import make_profile_hash, make_profile_group_kfold
from duplicate_group_weight import build_experiment_weight, effective_sample_size

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def resolve(raw, root=WORKSPACE_ROOT):
    if not raw:
        return None
    p = Path(raw)
    return p if p.is_absolute() else (root / p)

paths_cfg = cfg["paths"]
TRAIN_FEATURE_PATH = resolve(paths_cfg["train_feature_path"])
MANIFEST_PATH = resolve(cfg["features"]["manifest_path"])
GROUP5_FOLD_PATH = resolve(paths_cfg["group5_fold_assignment_path"])
GROUP5_PROVENANCE_PATH = resolve(paths_cfg["group5_fold_provenance_path"])
EXISTING_SKF_FOLD_PATH = resolve(paths_cfg["existing_skf_fold_assignment_path"])
ARTIFACT_ROOT = resolve(paths_cfg["artifact_root"]).resolve()

for label, p in {
    "train_feature_path": TRAIN_FEATURE_PATH, "manifest_path": MANIFEST_PATH,
    "group5_fold_assignment_path": GROUP5_FOLD_PATH,
    "group5_fold_provenance_path": GROUP5_PROVENANCE_PATH,
    "existing_skf_fold_assignment_path (건드리지 않고 존재만 확인)": EXISTING_SKF_FOLD_PATH,
}.items():
    if p is None or not p.exists():
        raise FileNotFoundError(f"{label} 파일이 없다: {p}")

_head = subprocess.run(["git", "-C", str(ONCO_AI_ROOT), "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
_dirty = bool(subprocess.run(
    ["git", "-C", str(ONCO_AI_ROOT), "status", "--porcelain", "--untracked-files=no"],
    capture_output=True, text=True, check=True).stdout.strip())
print(f"onco-ai HEAD={_head}, tracked_dirty={_dirty} (읽기 전용 확인 — 이 노트북은 team repo에 아무것도 쓰지 않는다)")
assert not _dirty, "onco-ai에 추적된 변경이 있다 — 이 실험은 팀 저장소를 읽기 전용으로 가정한다"

train_df = pd.read_parquet(TRAIN_FEATURE_PATH)
train_df["ID"] = train_df["ID"].astype(str)

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = yaml.safe_load(f)
GENE_ENCODING_COLUMNS = list(manifest["gene_encoding_feature_columns"])
OTHER_FEATURE_COLUMNS = list(manifest["sample_stateless_feature_columns"])
RATIO_TRANSFORM_FEATURE_COLUMNS = tuple(manifest["final_model_feature_columns"]["ratio_transform_output_columns"])

assert len(GENE_ENCODING_COLUMNS) == 4384
assert len(OTHER_FEATURE_COLUMNS) == 17
assert len(RATIO_TRANSFORM_FEATURE_COLUMNS) == 10
assert train_df.shape == (6201, 4411), f"train_df shape이 06과 다르다: {train_df.shape}"
assert "SUBCLASS" in train_df.columns
assert "ID" in train_df.columns

CLASS_ORDER = sorted(train_df["SUBCLASS"].unique().tolist())
N_CLASSES = len(CLASS_ORDER)
assert N_CLASSES == 26
CLASS_TO_LABEL = {c: i for i, c in enumerate(CLASS_ORDER)}
PROBA_COLUMNS = [f"proba__{c}" for c in CLASS_ORDER]
y_all_str = train_df["SUBCLASS"].to_numpy()
y_all_int = np.array([CLASS_TO_LABEL[c] for c in y_all_str])

print(f"train_df={train_df.shape}, 클래스 {N_CLASSES}개")
print(f"피처 구성: gene_encoding={len(GENE_ENCODING_COLUMNS)} + sample_stateless={len(OTHER_FEATURE_COLUMNS)} "
      f"+ ratio_transform={len(RATIO_TRANSFORM_FEATURE_COLUMNS)} = "
      f"{len(GENE_ENCODING_COLUMNS)+len(OTHER_FEATURE_COLUMNS)+len(RATIO_TRANSFORM_FEATURE_COLUMNS)} (06과 동일해야 함)")
print("데이터·피처 계약 검증 통과")

onco-ai HEAD=26f8cc571604dde387c24c2d8a208ef6358eb59f, tracked_dirty=False (읽기 전용 확인 — 이 노트북은 team repo에 아무것도 쓰지 않는다)


train_df=(6201, 4411), 클래스 26개
피처 구성: gene_encoding=4384 + sample_stateless=17 + ratio_transform=10 = 4411 (06과 동일해야 함)
데이터·피처 계약 검증 통과


## 7. Group5 fold 생성과 누수 검증

**이 셀이 하는 일**: 별도 스크립트(`scripts/build_group5_folds_xgb_enc3_sample27.py`)가 이미
생성해 둔 Group5 fold 파일과 provenance를 읽고, 이 노트북 안에서 **다시 한번** 독립적으로
누수 조건을 검증한다(스크립트를 신뢰만 하지 않는다).
**왜 필요한지**: 다섯 실험이 전부 같은 fold를 쓴다는 전제가 깨지면 비교 자체가 무의미하다.
**입력**: `fold_assignment__xgb_enc3_sample27_group5_s42.csv`, provenance JSON.
**출력**: `fold_assignment`(0-based, `train_df` 행 순서에 맞춘 배열), `profile_hash_all`.
**누수 방지 조건**: 그룹이 fold를 가로지르지 않는지, 원본 `train.csv`의 4,384개 문자열 열로
만든 해시인지(enc3 인코딩값이 아닌지)를 provenance로 재확인.
**확인해야 할 부분**: `groups_split_across_folds == 0`, 그룹 수·중복 그룹 수·최대 그룹 크기
출력값.

In [2]:
with open(GROUP5_PROVENANCE_PATH, "r", encoding="utf-8") as f:
    group5_provenance = json.load(f)

print("Group5 fold provenance:")
print(f"  seed={group5_provenance['seed']}, n_splits={group5_provenance['n_splits']}")
print(f"  cv_function={group5_provenance['cv_function']}")
print(f"  profile_hash_basis={group5_provenance['profile_hash_basis']['source']} "
      f"(gene_column_count={group5_provenance['profile_hash_basis']['gene_column_count']})")
print(f"  고유 그룹 {group5_provenance['n_unique_profile_hash_groups']}개, "
      f"중복 그룹(크기>1) {group5_provenance['n_duplicate_groups_size_gt_1']}개, "
      f"최대 그룹 크기 {group5_provenance['max_group_size']}")
print(f"  fold를 가로지른 그룹 수(provenance 기록): {group5_provenance['groups_split_across_folds']}")
assert group5_provenance["groups_split_across_folds"] == 0
assert group5_provenance["profile_hash_basis"]["gene_column_count"] == 4384
assert group5_provenance["seed"] == 42
assert group5_provenance["n_splits"] == 5

group5_df = pd.read_csv(GROUP5_FOLD_PATH, dtype={"ID": str, "profile_hash": str})
assert set(group5_df["ID"]) == set(train_df["ID"])
group5_df = group5_df.set_index("ID").loc[train_df["ID"]].reset_index()  # train_df 행 순서에 정렬

fold_assignment = group5_df["fold"].to_numpy()
profile_hash_all = group5_df["profile_hash"].to_numpy()
assert not pd.isna(fold_assignment).any()
assert set(fold_assignment.tolist()) == set(range(5))
N_SPLITS = 5

# 독립 재검증 — provenance를 신뢰만 하지 않고 이 노트북에서 다시 확인한다
check_df = pd.DataFrame({"ph": profile_hash_all, "fold": fold_assignment})
straddling = check_df.groupby("ph")["fold"].nunique()
n_straddling = int((straddling > 1).sum())
print(f"\n이 노트북에서 독립 재검증한 fold를 가로지른 그룹 수: {n_straddling}")
assert n_straddling == 0, "Group5 fold가 실제로는 그룹을 가로지르고 있다 — 중단"

print(f"fold별 행 수: {pd.Series(fold_assignment).value_counts().sort_index().to_dict()}")
print("기존 SKF fold 파일은 이 노트북에서 읽지도 쓰지도 않는다 — 완전히 별개 파일 사용 확인:")
print(f"  {GROUP5_FOLD_PATH.name}  (이번 실험)")
print(f"  {EXISTING_SKF_FOLD_PATH.name}  (06이 쓰던 것, 이번 실험은 건드리지 않음)")

Group5 fold provenance:
  seed=42, n_splits=5
  cv_function=cancer_hack.validation.make_profile_group_kfold (develop 병합본, StratifiedGroupKFold + groups=make_profile_hash 문자열)
  profile_hash_basis=원본 train.csv의 4,384개 변이 문자열 열 (dtype=str, na_filter=False로 읽음, 인코딩/가공 없음) (gene_column_count=4384)
  고유 그룹 5636개, 중복 그룹(크기>1) 451개, 최대 그룹 크기 94
  fold를 가로지른 그룹 수(provenance 기록): 0

이 노트북에서 독립 재검증한 fold를 가로지른 그룹 수: 0
fold별 행 수: {0: 1161, 1: 1206, 2: 1233, 3: 1249, 4: 1352}
기존 SKF fold 파일은 이 노트북에서 읽지도 쓰지도 않는다 — 완전히 별개 파일 사용 확인:
  fold_assignment__xgb_enc3_sample27_group5_s42.csv  (이번 실험)
  fold_assignment__lr_rf_enc3_sample27_full_skf5_s42.csv  (06이 쓰던 것, 이번 실험은 건드리지 않음)


## 8. 작은 예제로 weight 계산 확인

**이 셀이 하는 일**: 실제 데이터를 쓰기 전에, 손으로 검산 가능한 작은 합성 예제(9행,
3클래스)로 C/D/E 세 방식이 기대한 대로 다른 값을 내는지 확인한다. 이 예제는
`lib/test_duplicate_group_weight.py`의 단위테스트와 동일한 예제다.
**왜 필요한지**: 실제 6,201행 위에서 숫자가 이상해도 원인을 찾기 어렵다 — 작은 예제로
"프로필 p1이 라벨 A,A,B로 갈라지는 경우" 같은 핵심 케이스를 먼저 눈으로 확인한다.
**입력**: 합성 배열 `Y`, `PH`(9행).
**출력**: 스킴별 가중치 표.
**누수 방지 조건**: 해당 없음(합성 데이터, fold 개념 없음).
**확인해야 할 부분**: `p1` 그룹(행 0,1,3)이 profile-only로는 하나의 그룹(크기3)이지만
same-label로는 `(A,p1)`(크기2)와 `(B,p1)`(크기1)로 갈라지는 것.

In [3]:
Y_TOY = np.array(["A", "A", "A", "B", "B", "C", "C", "C", "C"])
PH_TOY = np.array(["p1", "p1", "p2", "p1", "p3", "p4", "p4", "p4", "p5"])

toy_rows = []
for scheme in ["A_none", "B_balanced", "C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    result = build_experiment_weight(scheme, Y_TOY, PH_TOY)
    w = result["normalized"]
    for i in range(len(Y_TOY)):
        toy_rows.append({
            "row": i, "y": Y_TOY[i], "profile_hash": PH_TOY[i], "scheme": scheme,
            "weight": None if w is None else round(float(w[i]), 4),
        })
toy_df = pd.DataFrame(toy_rows).pivot(index=["row", "y", "profile_hash"], columns="scheme", values="weight")
print(toy_df.to_string())
print("\n확인: row0/1(라벨A,프로필p1)과 row3(라벨B,프로필p1) — profile-only(C)는 이 셋을 같은 분모(3)로")
print("나누지만, same-label(D,E)은 row0/1(분모2)과 row3(분모1)을 따로 취급한다.")
c_row0 = build_experiment_weight("C_balanced_profile", Y_TOY, PH_TOY)["normalized"][0]
d_row0 = build_experiment_weight("D_balanced_same_label_naive", Y_TOY, PH_TOY)["normalized"][0]
print(f"\nrow0의 profile-only 기반 가중치(C)={c_row0:.4f} vs same-label 기반 가중치(D)={d_row0:.4f} — 다르면 정상(그룹 정의가 다르므로)")
assert not np.isclose(c_row0, d_row0), "C와 D가 이 예제에서 같은 값을 내면 그룹 정의 구현이 잘못된 것"
print("작은 예제 검산 통과")

scheme              A_none  B_balanced  C_balanced_profile  D_balanced_same_label_naive  E_same_label_rebalanced
row y profile_hash                                                                                              
0   A p1               NaN        1.00              0.5806                       0.6923                     0.75
1   A p1               NaN        1.00              0.5806                       0.6923                     0.75
2   A p2               NaN        1.00              1.7419                       1.3846                     1.50
3   B p1               NaN        1.50              0.8710                       2.0769                     1.50
4   B p3               NaN        1.50              2.6129                       2.0769                     1.50
5   C p4               NaN        0.75              0.4355                       0.3462                     0.50
6   C p4               NaN        0.75              0.4355                       0.3462         

## 9. 실제 데이터 weight 분포 (fold-local 계산)

**이 셀이 하는 일**: Group5의 5개 fold 각각에 대해 **그 fold의 train 부분만**으로 B/C/D/E
가중치를 계산하고, raw(정규화 전)와 normalized(평균1) 분포를 함께 기록한다. 결과는
`WEIGHT_CACHE`에 캐싱해 10절(모델 학습)에서 재사용한다(같은 계산을 두 번 하지 않되,
반드시 fold-local로만 계산됐음을 보장한다).
**왜 필요한지**: "전체 train에서 미리 계산해 fold별로 잘라 쓰는" 금지된 방식과 지금 이
방식이 다르다는 것을 코드로 증명해야 한다 — 이 셀은 fold의 train 인덱스만 받아 그 안에서
그룹 크기를 다시 센다.
**입력**: `fold_assignment`, `profile_hash_all`, `y_all_str` (fold별 train 슬라이스만 사용).
**출력**: `WEIGHT_CACHE[(fold_idx, scheme)]`, 분포 요약표(`weight_diagnostics_df`).
**누수 방지 조건**: validation 인덱스는 이 셀의 어떤 계산에도 들어가지 않는다 —
`train_idx = np.where(fold_assignment != fold_idx)[0]`만 쓴다.
**확인해야 할 부분**: 각 스킴의 normalized 평균이 1인지, E의 클래스별 총weight가 N/K와
일치하는지, D에서 클래스별 총weight가 실제로 불균등한지(우려가 실측되는지).

In [4]:
WEIGHT_CACHE = {}
weight_diag_rows = []
per_class_total_rows = []

for fold_idx in range(N_SPLITS):
    train_idx = np.where(fold_assignment != fold_idx)[0]
    val_idx = np.where(fold_assignment == fold_idx)[0]
    # validation 인덱스는 절대 아래 어떤 계산에도 넘기지 않는다
    y_train_fold = y_all_str[train_idx]
    ph_train_fold = profile_hash_all[train_idx]

    for scheme in ["A_none", "B_balanced", "C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
        result = build_experiment_weight(scheme, y_train_fold, ph_train_fold)
        WEIGHT_CACHE[(fold_idx, scheme)] = result
        if result["normalized"] is None:
            continue
        w = result["normalized"]
        raw = result["raw"]
        ess = result["effective_sample_size"]
        weight_diag_rows.append({
            "fold": fold_idx, "scheme": scheme, "n": result["n"], "k": result["k"],
            "raw_mean": float(raw.mean()), "raw_min": float(raw.min()), "raw_max": float(raw.max()),
            "norm_mean": float(w.mean()), "norm_std": float(w.std()),
            "norm_min": float(w.min()), "norm_max": float(w.max()),
            "norm_p1": float(np.percentile(w, 1)), "norm_p50": float(np.percentile(w, 50)),
            "norm_p99": float(np.percentile(w, 99)),
            "effective_sample_size": ess, "ess_ratio": ess / result["n"],
        })
        totals = np.array(list(result["per_class_total_weight"].values()))
        per_class_total_rows.append({
            "fold": fold_idx, "scheme": scheme,
            "class_total_weight_mean": float(totals.mean()),
            "class_total_weight_std": float(totals.std()),
            "class_total_weight_min": float(totals.min()),
            "class_total_weight_max": float(totals.max()),
            "expected_n_over_k": result["n"] / result["k"],
        })

weight_diagnostics_df = pd.DataFrame(weight_diag_rows)
per_class_total_df = pd.DataFrame(per_class_total_rows)

print("스킴별 fold-평균 weight 분포(정규화 후):")
print(weight_diagnostics_df.groupby("scheme")[["norm_mean", "norm_std", "norm_min", "norm_max", "effective_sample_size", "ess_ratio"]].mean().round(4).to_string())

print("\n스킴별 클래스별 총weight 균형(fold-평균) — std가 0에 가까우면 균등, 크면 불균등:")
print(per_class_total_df.groupby("scheme")[["class_total_weight_std", "class_total_weight_min", "class_total_weight_max"]].mean().round(4).to_string())

# B와 E는 반드시 평균이 1이어야 한다
for scheme in ["B_balanced", "E_same_label_rebalanced"]:
    means = weight_diagnostics_df.loc[weight_diagnostics_df["scheme"] == scheme, "norm_mean"]
    assert np.allclose(means, 1.0, atol=1e-6), f"{scheme}의 평균이 1이 아니다: {means.tolist()}"

# E는 클래스별 총weight std가 0에 가까워야 한다(수식 자체가 보장)
e_std = per_class_total_df.loc[per_class_total_df["scheme"] == "E_same_label_rebalanced", "class_total_weight_std"]
assert (e_std < 1e-6).all(), f"E의 클래스별 총weight가 균등하지 않다: {e_std.tolist()}"
print("\nE 방식 클래스별 총weight 균형 검증 통과(std < 1e-6)")

d_std = per_class_total_df.loc[per_class_total_df["scheme"] == "D_balanced_same_label_naive", "class_total_weight_std"]
print(f"D 방식(Notion 원안) 클래스별 총weight std(fold평균) = {d_std.mean():.4f} — 0보다 뚜렷이 크면 우려가 실측된 것")

스킴별 fold-평균 weight 분포(정규화 후):
                             norm_mean  norm_std  norm_min  norm_max  effective_sample_size  ess_ratio
scheme                                                                                                
B_balanced                         1.0    0.7020    0.3035    6.3009              3323.0940     0.6699
C_balanced_profile                 1.0    0.7906    0.0344    6.7959              3052.5984     0.6154
D_balanced_same_label_naive        1.0    0.7201    0.0779    6.4462              3266.5415     0.6586
E_same_label_rebalanced            1.0    0.7345    0.0815    6.3009              3221.9738     0.6496

스킴별 클래스별 총weight 균형(fold-평균) — std가 0에 가까우면 균등, 크면 불균등:
                             class_total_weight_std  class_total_weight_min  class_total_weight_max
scheme                                                                                             
B_balanced                                   0.0000                190.8000                190.

## 10. A~E 모델 학습

**이 셀이 하는 일**: 5개 fold × 5개 스킴 = 25회 XGBoost 학습을 수행한다. 각 fold 안에서
`RatioTransformFeatures`는 train fold에만 `fit`하고(누수 차단), 9절에서 미리 계산해 둔
fold-local weight를 그대로 재사용한다. `test`는 이번 실험에서 아예 로드하지 않는다(제출
예측을 만들 필요가 없으므로).
**왜 필요한지**: 이게 이 실험의 핵심 산출물 — 스킴별 OOF 확률을 전부 모은다.
**입력**: `train_df`, `fold_assignment`, `WEIGHT_CACHE`.
**출력**: `OOF_PROBA[scheme]`(6201×26), `fold_metrics_rows`, `all_warnings`.
**누수 방지 조건**: (1) `RatioTransformFeatures.fit`은 train_idx에만, val_idx는 `transform`만.
(2) `sample_weight`는 `WEIGHT_CACHE[(fold_idx, scheme)]`에서 그대로 가져온다(다시 계산하지
않음 — 9절과 정확히 같은 값이라는 뜻). (3) `model.fit`에 `eval_set`을 주지 않는다(이른
종료에 val을 안 씀). (4) test는 이 셀에서 아예 참조되지 않는다.
**확인해야 할 부분**: 다섯 스킴이 완전히 같은 fold·같은 하이퍼파라미터를 쓰는지(코드 상
`XGB_PARAMS`와 `fold_assignment`가 스킴 루프 밖에서 고정된 채 재사용되는 것으로 확인),
fold별 학습 시간, warning 발생 여부.

In [5]:
def assemble_sparse(gene_block, other_block, ratio_block):
    all_cols = list(gene_block.columns) + list(other_block.columns) + list(ratio_block.columns)
    assert len(set(all_cols)) == len(all_cols), "조립 블록 사이에 중복 컬럼이 있다"
    gs = sp.csr_matrix(gene_block.to_numpy(dtype="float32"))
    os_ = sp.csr_matrix(other_block.to_numpy(dtype="float32"))
    rs = sp.csr_matrix(ratio_block.to_numpy(dtype="float32"))
    return sp.hstack([gs, os_, rs], format="csr")

XGB_PARAMS = dict(cfg["xgboost"])  # 다섯 스킴, 다섯 fold 전부 이 하나의 dict를 그대로 쓴다 — 스킴별 튜닝 없음

def build_xgb_model():
    return xgb.XGBClassifier(num_class=N_CLASSES, **XGB_PARAMS)

SCHEMES = ["A_none", "B_balanced", "C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]
OOF_PROBA = {scheme: np.full((len(train_df), N_CLASSES), np.nan, dtype="float64") for scheme in SCHEMES}
fold_metrics_rows = []
all_warnings = []
run_started = time.perf_counter()

for fold_idx in range(N_SPLITS):
    fold_started = time.perf_counter()
    train_idx = np.where(fold_assignment != fold_idx)[0]
    val_idx = np.where(fold_assignment == fold_idx)[0]

    rtf = RatioTransformFeatures()
    train_ratio = rtf.fit_transform(train_df.iloc[train_idx])   # train fold에만 fit
    val_ratio = rtf.transform(train_df.iloc[val_idx])           # transform만

    X_train = assemble_sparse(train_df.iloc[train_idx][GENE_ENCODING_COLUMNS],
                               train_df.iloc[train_idx][OTHER_FEATURE_COLUMNS], train_ratio)
    X_val = assemble_sparse(train_df.iloc[val_idx][GENE_ENCODING_COLUMNS],
                             train_df.iloc[val_idx][OTHER_FEATURE_COLUMNS], val_ratio)
    y_train = y_all_int[train_idx]

    for scheme in SCHEMES:
        scheme_started = time.perf_counter()
        weight_result = WEIGHT_CACHE[(fold_idx, scheme)]
        sample_weight = weight_result["normalized"]  # None(A) 또는 fold-local 정규화된 배열

        model = build_xgb_model()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(X_train, y_train, sample_weight=sample_weight)  # eval_set 없음
        for w in caught:
            all_warnings.append({"fold": fold_idx, "scheme": scheme, "category": w.category.__name__, "message": str(w.message)})

        val_proba = model.predict_proba(X_val)
        OOF_PROBA[scheme][val_idx] = val_proba

        val_pred = np.array(CLASS_ORDER)[val_proba.argmax(axis=1)]
        fold_f1 = f1_score(y_all_str[val_idx], val_pred, average="macro")
        elapsed = time.perf_counter() - scheme_started
        fold_metrics_rows.append({"fold": int(fold_idx), "scheme": scheme, "macro_f1": float(fold_f1), "elapsed_sec": elapsed})
        print(f"[fold {fold_idx}][{scheme}] macro_f1={fold_f1:.4f}  {elapsed:.1f}s")

    print(f"-- fold {fold_idx} 전체(5스킴) 소요: {time.perf_counter()-fold_started:.1f}s --")

total_elapsed = time.perf_counter() - run_started
print(f"\n총 소요 시간: {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")

for scheme in SCHEMES:
    assert not np.isnan(OOF_PROBA[scheme]).any(), f"{scheme}의 OOF에 채워지지 않은 행이 있다"
    assert np.allclose(OOF_PROBA[scheme].sum(axis=1), 1.0, atol=1e-3), f"{scheme}의 OOF 확률 합이 1이 아니다"
    assert OOF_PROBA[scheme].shape == (len(train_df), 26)
print("\nOOF 완전성 검증 통과: 결측 없음, 확률합 ~1, 26개 클래스 컬럼")

[fold 0][A_none] macro_f1=0.3816  21.0s


[fold 0][B_balanced] macro_f1=0.4093  22.3s


[fold 0][C_balanced_profile] macro_f1=0.3978  21.2s


[fold 0][D_balanced_same_label_naive] macro_f1=0.4033  26.0s


[fold 0][E_same_label_rebalanced] macro_f1=0.4077  29.7s
-- fold 0 전체(5스킴) 소요: 120.5s --


...

[fold 4][A_none] macro_f1=0.3931  25.3s


[fold 4][B_balanced] macro_f1=0.3878  24.1s


[fold 4][C_balanced_profile] macro_f1=0.3821  32.6s


[fold 4][D_balanced_same_label_naive] macro_f1=0.3838  50.7s


[fold 4][E_same_label_rebalanced] macro_f1=0.3844  38.2s
-- fold 4 전체(5스킴) 소요: 171.2s --

총 소요 시간: 655.2초 (10.9분)

OOF 완전성 검증 통과: 결측 없음, 확률합 ~1, 26개 클래스 컬럼


## 11. 전체 OOF 비교표

**이 셀이 하는 일**: 6,201행 전체 OOF에서 스킴별 Macro F1을 계산하고, A/B 대비 개선폭을
표로 만든다.
**왜 필요한지**: 이게 이 실험의 주지표(§10 사전 판정 기준의 핵심 입력).
**입력**: `OOF_PROBA`.
**출력**: `oof_summary_df`.
**누수 방지 조건**: 해당 없음(집계만).
**확인해야 할 부분**: B 대비 C/D/E의 개선폭이 0.005를 넘는지, A 대비 B의 순수 클래스균형
효과는 얼마인지.

In [6]:
oof_pred = {scheme: np.array(CLASS_ORDER)[OOF_PROBA[scheme].argmax(axis=1)] for scheme in SCHEMES}
oof_macro_f1 = {scheme: f1_score(y_all_str, oof_pred[scheme], average="macro") for scheme in SCHEMES}

oof_summary_rows = []
baseline_b = oof_macro_f1["B_balanced"]
baseline_a = oof_macro_f1["A_none"]
for scheme in SCHEMES:
    oof_summary_rows.append({
        "scheme": scheme,
        "oof_macro_f1": oof_macro_f1[scheme],
        "delta_vs_A": oof_macro_f1[scheme] - baseline_a,
        "delta_vs_B": oof_macro_f1[scheme] - baseline_b,
    })
oof_summary_df = pd.DataFrame(oof_summary_rows).sort_values("oof_macro_f1", ascending=False)
print(oof_summary_df.to_string(index=False))

                     scheme  oof_macro_f1  delta_vs_A  delta_vs_B
                 B_balanced      0.407017    0.009910    0.000000
         C_balanced_profile      0.405309    0.008202   -0.001708
    E_same_label_rebalanced      0.404210    0.007103   -0.002807
D_balanced_same_label_naive      0.403271    0.006164   -0.003746
                     A_none      0.397107    0.000000   -0.009910


## 12. fold별 비교

**이 셀이 하는 일**: fold별 Macro F1과 평균±표준편차를 스킴별로 비교한다.
**왜 필요한지**: 전체 OOF가 좋아져도 특정 fold 하나에 몰린 결과라면 신뢰할 수 없다 —
사전 판정 기준의 "5개 fold 중 최소 3개에서 개선"을 확인하는 데 쓴다.
**입력**: `fold_metrics_rows`.
**출력**: `fold_pivot_df`, `n_folds_improved_vs_B`.
**누수 방지 조건**: 해당 없음(집계만).
**확인해야 할 부분**: 표준편차가 A/B 대비 급격히 커진 스킴이 있는지.

In [7]:
fold_metrics_df = pd.DataFrame(fold_metrics_rows)
fold_pivot_df = fold_metrics_df.pivot(index="fold", columns="scheme", values="macro_f1")[SCHEMES]
print(fold_pivot_df.round(4).to_string())

fold_mean_std_df = fold_metrics_df.groupby("scheme")["macro_f1"].agg(["mean", "std"]).loc[SCHEMES]
print("\nfold 평균 ± 표준편차:")
print(fold_mean_std_df.round(4).to_string())

n_folds_improved_vs_B = {}
for scheme in ["C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    improved = (fold_pivot_df[scheme] > fold_pivot_df["B_balanced"]).sum()
    n_folds_improved_vs_B[scheme] = int(improved)
    print(f"{scheme}: B 대비 개선된 fold 수 = {improved}/5")

scheme  A_none  B_balanced  C_balanced_profile  D_balanced_same_label_naive  E_same_label_rebalanced
fold                                                                                                
0       0.3816      0.4093              0.3978                       0.4033                   0.4077
1       0.4006      0.3898              0.3925                       0.3816                   0.3843
2       0.3860      0.4094              0.4030                       0.4108                   0.3996
3       0.4066      0.4230              0.4303                       0.4224                   0.4254
4       0.3931      0.3878              0.3821                       0.3838                   0.3844

fold 평균 ± 표준편차:
                               mean     std
scheme                                     
A_none                       0.3936  0.0102
B_balanced                   0.4039  0.0148
C_balanced_profile           0.4012  0.0180
D_balanced_same_label_naive  0.4004  0.0175
E_same_label

## 13. 클래스별 비교

**이 셀이 하는 일**: 26개 클래스별 F1을 스킴별로 계산하고, A 대비 가장 크게 오르거나
내린 클래스를 찾는다.
**왜 필요한지**: 전체 Macro F1이 올라도 소수 클래스 몇 개만 오르고 다수가 붕괴하면
사전 판정 기준(§11)에 따라 보류해야 한다.
**입력**: `OOF_PROBA`, `y_all_str`.
**출력**: `per_class_f1_df`.
**누수 방지 조건**: 해당 없음(집계만).
**확인해야 할 부분**: `E`가 소수 클래스를 특별히 해치지는 않는지.

In [8]:
per_class_f1_rows = []
for scheme in SCHEMES:
    f1_values = f1_score(y_all_str, oof_pred[scheme], average=None, labels=CLASS_ORDER)
    for cls, f1v in zip(CLASS_ORDER, f1_values):
        per_class_f1_rows.append({"scheme": scheme, "class": cls, "f1": f1v})
per_class_f1_df = pd.DataFrame(per_class_f1_rows).pivot(index="class", columns="scheme", values="f1")[SCHEMES]
class_counts = pd.Series(y_all_str).value_counts()
per_class_f1_df.insert(0, "n", class_counts.reindex(per_class_f1_df.index))

for scheme in ["C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    per_class_f1_df[f"delta_{scheme}_vs_A"] = per_class_f1_df[scheme] - per_class_f1_df["A_none"]

print("클래스별 F1(상위 스킴 E 기준 변화 큰 순):")
print(per_class_f1_df.sort_values("delta_E_same_label_rebalanced_vs_A").round(4).to_string())

클래스별 F1(상위 스킴 E 기준 변화 큰 순):
scheme    n  A_none  B_balanced  C_balanced_profile  D_balanced_same_label_naive  E_same_label_rebalanced  delta_C_balanced_profile_vs_A  delta_D_balanced_same_label_naive_vs_A  delta_E_same_label_rebalanced_vs_A
class                                                                                                                                                                                                               
DLBC     38  0.4000      0.3265              0.2979                       0.3333                   0.2917                        -0.1021                                 -0.0667                             -0.1083
KIPAN   515  0.4046      0.3491              0.3538                       0.3457                   0.3337                        -0.0508                                 -0.0590                             -0.0709
GBMLGG  461  0.4661      0.4168              0.4172                       0.4110                   0.4211               

## 14. singleton / duplicate 비교

**이 셀이 하는 일**: 전체 train 기준 `profile_hash` 그룹 크기가 1인 행(singleton)과 2 이상인
행(duplicate)을 나눠, 스킴별로 singleton-only OOF Macro F1과 duplicate-only OOF Macro F1을
따로 계산한다. A 대비 예측 불일치율도 함께 본다.
**왜 필요한지**: 전체 OOF가 좋아져도 그게 singleton 성능을 깎아 먹은 대가라면(사전 판정
기준 §11) 보류 대상이다. singleton은 진단용이며 전체 OOF를 대체하지 않는다.
**입력**: `profile_hash_all`(전체 train 기준, fold와 무관), `OOF_PROBA`.
**출력**: `singleton_mask`, `singleton_vs_duplicate_df`, `disagreement_df`.
**누수 방지 조건**: 이 그룹 크기는 fold 계산과 무관한 診단 지표라 fold-local일 필요가
없다 — 전체 train에서 "이 프로필이 몇 번 나오는가"라는 데이터 성질을 재는 것뿐이다.
**확인해야 할 부분**: singleton-only Macro F1이 0.005 넘게 악화된 스킴이 있는지.

In [9]:
profile_group_size_full = pd.Series(profile_hash_all).map(pd.Series(profile_hash_all).value_counts())
singleton_mask = (profile_group_size_full == 1).to_numpy()
duplicate_mask = ~singleton_mask
print(f"singleton 행: {singleton_mask.sum()} / {len(train_df)} ({singleton_mask.mean()*100:.2f}%)")
print(f"duplicate 행: {duplicate_mask.sum()} / {len(train_df)} ({duplicate_mask.mean()*100:.2f}%)")

singleton_dup_rows = []
for scheme in SCHEMES:
    singleton_f1 = f1_score(y_all_str[singleton_mask], oof_pred[scheme][singleton_mask], average="macro")
    duplicate_f1 = f1_score(y_all_str[duplicate_mask], oof_pred[scheme][duplicate_mask], average="macro")
    singleton_dup_rows.append({
        "scheme": scheme, "singleton_macro_f1": singleton_f1, "duplicate_macro_f1": duplicate_f1,
        "n_singleton": int(singleton_mask.sum()), "n_duplicate": int(duplicate_mask.sum()),
    })
singleton_vs_duplicate_df = pd.DataFrame(singleton_dup_rows)
singleton_vs_duplicate_df["delta_singleton_vs_A"] = singleton_vs_duplicate_df["singleton_macro_f1"] - singleton_vs_duplicate_df.loc[0, "singleton_macro_f1"]
print(singleton_vs_duplicate_df.round(4).to_string(index=False))

disagreement_rows = []
for scheme in SCHEMES[1:]:
    rate = float((oof_pred[scheme] != oof_pred["A_none"]).mean())
    disagreement_rows.append({"scheme": scheme, "disagreement_rate_vs_A": rate})
disagreement_df = pd.DataFrame(disagreement_rows)
print("\nA(가중치 없음) 대비 예측 불일치율:")
print(disagreement_df.round(4).to_string(index=False))

cm_a = confusion_matrix(y_all_str, oof_pred["A_none"], labels=CLASS_ORDER)
cm_e = confusion_matrix(y_all_str, oof_pred["E_same_label_rebalanced"], labels=CLASS_ORDER)
cm_diff = cm_e - cm_a
np.fill_diagonal(cm_diff, 0)
flat_idx = np.argsort(np.abs(cm_diff).ravel())[::-1][:8]
print("\nA -> E로 바뀔 때 가장 크게 변한 오혼동 쌍(양수=E에서 늘어남, 음수=E에서 줄어듦):")
for idx in flat_idx:
    i, j = divmod(idx, len(CLASS_ORDER))
    if cm_diff[i, j] != 0:
        print(f"  {CLASS_ORDER[i]:8s} -> {CLASS_ORDER[j]:8s}: {cm_diff[i,j]:+d}건")

singleton 행: 5185 / 6201 (83.62%)
duplicate 행: 1016 / 6201 (16.38%)
                     scheme  singleton_macro_f1  duplicate_macro_f1  n_singleton  n_duplicate  delta_singleton_vs_A
                     A_none              0.3890              0.1293         5185         1016                0.0000
                 B_balanced              0.4060              0.1210         5185         1016                0.0170
         C_balanced_profile              0.4075              0.1195         5185         1016                0.0185
D_balanced_same_label_naive              0.4031              0.1208         5185         1016                0.0141
    E_same_label_rebalanced              0.4048              0.1145         5185         1016                0.0159

A(가중치 없음) 대비 예측 불일치율:
                     scheme  disagreement_rate_vs_A
                 B_balanced                  0.2459
         C_balanced_profile                  0.2546
D_balanced_same_label_naive                  0.2475
    E

## 15. 알게 된 점

아래 셀은 지금까지 계산된 실제 수치로부터 6가지 핵심 질문에 대한 답을 **코드로 직접
도출**한다 — 사람이 나중에 손으로 요약해 끼워 넣는 게 아니라, 노트북을 다시 실행해도
같은 논리로 같은 결론이 나오게 한다.

In [10]:
print("=" * 70)
print("질문 1: 중복 가중치가 무가중 모델(A)보다 실제로 좋아졌는가?")
print("=" * 70)
for scheme in ["C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    delta = oof_summary_df.set_index("scheme").loc[scheme, "delta_vs_A"]
    print(f"  {scheme}: A 대비 {delta:+.4f} ({'개선' if delta > 0 else '악화 또는 동일'})")

print()
print("=" * 70)
print("질문 2: 클래스 균형 모델(B)보다도 좋아졌는가? (사전 판정 기준: >=0.005, 5개 중 3개 fold 이상)")
print("=" * 70)
for scheme in ["C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    delta_b = oof_summary_df.set_index("scheme").loc[scheme, "delta_vs_B"]
    n_improved = n_folds_improved_vs_B[scheme]
    meets_bar = delta_b >= 0.005
    meets_fold = n_improved >= 3
    print(f"  {scheme}: B 대비 {delta_b:+.4f} ({'0.005 이상' if meets_bar else '0.005 미만(근소/불확실)'}), "
          f"fold 개선 {n_improved}/5 ({'충족' if meets_fold else '미충족'})")

print()
print("=" * 70)
print("질문 3: profile-only(C)와 same-label(D/E) 중 어느 정의가 더 나았는가?")
print("=" * 70)
f1_c = oof_macro_f1["C_balanced_profile"]
f1_d = oof_macro_f1["D_balanced_same_label_naive"]
f1_e = oof_macro_f1["E_same_label_rebalanced"]
print(f"  C(profile-only)={f1_c:.4f}, D(same-label naive)={f1_d:.4f}, E(same-label 교정)={f1_e:.4f}")
same_label_better = max(f1_d, f1_e) > f1_c
print(f"  -> {'same-label 계열이 더 나았다' if same_label_better else 'profile-only(C)가 더 나았거나 비슷했다'}")

print()
print("=" * 70)
print("질문 4: Notion 단순곱(D)의 클래스 불균형 문제가 실제 성능에도 영향을 주었는가?")
print("=" * 70)
d_class_std = per_class_total_df.loc[per_class_total_df['scheme'] == 'D_balanced_same_label_naive', 'class_total_weight_std'].mean()
e_class_std = per_class_total_df.loc[per_class_total_df['scheme'] == 'E_same_label_rebalanced', 'class_total_weight_std'].mean()
print(f"  D의 클래스별 총weight std(fold평균)={d_class_std:.4f} vs E={e_class_std:.6f} (E는 설계상 0에 근접해야 함)")
d_vs_e_gap = f1_d - f1_e
print(f"  D와 E의 OOF Macro F1 차이: {d_vs_e_gap:+.4f} ({'D가 실제로 더 나쁘다 -> 불균형이 성능에 반영됨' if d_vs_e_gap < -0.002 else 'D와 E 성능 차이가 뚜렷하지 않다 -> 불균형이 있어도 이 데이터/모델 조합에선 성능 영향이 작을 수 있음'})")

print()
print("=" * 70)
print("질문 5: 교정식(E)이 문제를 해결하면서 성능도 유지했는가?")
print("=" * 70)
print(f"  E의 클래스별 총weight 균형: std={e_class_std:.6f} (문제 해결 확인)")
print(f"  E의 OOF Macro F1={f1_e:.4f} vs B(클래스균형만)={baseline_b:.4f}, 차이 {f1_e-baseline_b:+.4f}")
e_singleton_delta = singleton_vs_duplicate_df.set_index('scheme').loc['E_same_label_rebalanced', 'delta_singleton_vs_A']
print(f"  E의 singleton-only OOF는 A 대비 {e_singleton_delta:+.4f} ({'0.005 넘게 악화되지 않음 -> 안전' if e_singleton_delta > -0.005 else '0.005 넘게 악화 -> 주의 필요'})")

print()
print("=" * 70)
print("질문 6: 최종적으로 개인 모델에 어떤 방식을 채택해야 하는가? -> 16절에서 기준별로 판정")
print("=" * 70)

질문 1: 중복 가중치가 무가중 모델(A)보다 실제로 좋아졌는가?
  C_balanced_profile: A 대비 +0.0082 (개선)
  D_balanced_same_label_naive: A 대비 +0.0062 (개선)
  E_same_label_rebalanced: A 대비 +0.0071 (개선)

질문 2: 클래스 균형 모델(B)보다도 좋아졌는가? (사전 판정 기준: >=0.005, 5개 중 3개 fold 이상)
  C_balanced_profile: B 대비 -0.0017 (0.005 미만(근소/불확실)), fold 개선 2/5 (미충족)
  D_balanced_same_label_naive: B 대비 -0.0037 (0.005 미만(근소/불확실)), fold 개선 1/5 (미충족)
  E_same_label_rebalanced: B 대비 -0.0028 (0.005 미만(근소/불확실)), fold 개선 1/5 (미충족)

질문 3: profile-only(C)와 same-label(D/E) 중 어느 정의가 더 나았는가?
  C(profile-only)=0.4053, D(same-label naive)=0.4033, E(same-label 교정)=0.4042
  -> profile-only(C)가 더 나았거나 비슷했다

질문 4: Notion 단순곱(D)의 클래스 불균형 문제가 실제 성능에도 영향을 주었는가?
  D의 클래스별 총weight std(fold평균)=11.6989 vs E=0.000000 (E는 설계상 0에 근접해야 함)
  D와 E의 OOF Macro F1 차이: -0.0009 (D와 E 성능 차이가 뚜렷하지 않다 -> 불균형이 있어도 이 데이터/모델 조합에선 성능 영향이 작을 수 있음)

질문 5: 교정식(E)이 문제를 해결하면서 성능도 유지했는가?
  E의 클래스별 총weight 균형: std=0.000000 (문제 해결 확인)
  E의 OOF Macro F1=0.4042 vs B(클래스균형만)=0.4070, 차이 -0.0028
  

## 16. 채택·보류·기각 판정

**이 셀이 하는 일**: §11의 사전 판정 기준(B 대비 0.005 이상 개선, 5개 중 3개 이상 fold
개선, singleton-only 0.005 넘게 악화 금지, fold 표준편차 급증 금지, 특정 소수 클래스만
개선되고 다수가 붕괴하지 않을 것)을 C/D/E 각각에 대해 **코드로** 적용해 판정한다.
**왜 필요한지**: "OOF가 조금 올랐다"는 이유만으로 자동 채택하지 않기 위해 기준을
미리 고정하고 그대로 적용한다.
**입력**: 11~14절의 모든 집계 결과.
**출력**: `verdict_df`.
**누수 방지 조건**: 해당 없음(판정 로직).
**확인해야 할 부분**: D가 E보다 OOF가 좋아도 클래스 불균형 구조적 위험은 별도로 명시하는지.

In [11]:
def judge(scheme):
    delta_b = oof_summary_df.set_index("scheme").loc[scheme, "delta_vs_B"]
    n_improved = n_folds_improved_vs_B[scheme]
    singleton_delta = singleton_vs_duplicate_df.set_index("scheme").loc[scheme, "delta_singleton_vs_A"]
    fold_std = fold_mean_std_df.loc[scheme, "std"]
    b_fold_std = fold_mean_std_df.loc["B_balanced", "std"]
    class_deltas = per_class_f1_df[f"delta_{scheme}_vs_A"] if f"delta_{scheme}_vs_A" in per_class_f1_df.columns else None

    reasons = []
    meets_improvement = delta_b >= 0.005
    reasons.append(f"B 대비 개선폭 {delta_b:+.4f} ({'충족' if meets_improvement else '미충족 — 0.005 미만'})")
    meets_fold_count = n_improved >= 3
    reasons.append(f"fold 개선 {n_improved}/5 ({'충족' if meets_fold_count else '미충족'})")
    meets_singleton = singleton_delta > -0.005
    reasons.append(f"singleton-only delta {singleton_delta:+.4f} ({'충족(악화 아님)' if meets_singleton else '미충족 — 0.005 넘게 악화'})")
    fold_std_ratio = fold_std / b_fold_std if b_fold_std > 0 else float('inf')
    meets_stability = fold_std_ratio < 2.0
    reasons.append(f"fold 표준편차 {fold_std:.4f} (B의 {fold_std_ratio:.2f}배, {'충족' if meets_stability else '미충족 — 급증'})")

    collapse_flag = False
    if class_deltas is not None:
        n_improved_classes = (class_deltas > 0.01).sum()
        n_collapsed_classes = (class_deltas < -0.05).sum()
        collapse_flag = n_collapsed_classes >= 3 and n_improved_classes <= 2
        reasons.append(f"클래스 붕괴 점검: 0.05 넘게 악화된 클래스 {n_collapsed_classes}개, "
                        f"0.01 넘게 개선된 클래스 {n_improved_classes}개 ({'붕괴 패턴 의심' if collapse_flag else '정상 범위'})")

    if not meets_improvement:
        verdict = "기각(개선폭 부족)" if delta_b < 0 else "보류(근소/불확실)"
    elif not meets_fold_count or not meets_singleton or not meets_stability or collapse_flag:
        verdict = "보류(개선은 있으나 안정성/부작용 조건 미충족)"
    else:
        verdict = "채택 후보"
    return verdict, reasons

verdict_rows = []
for scheme in ["C_balanced_profile", "D_balanced_same_label_naive", "E_same_label_rebalanced"]:
    verdict, reasons = judge(scheme)
    verdict_rows.append({"scheme": scheme, "verdict": verdict})
    print(f"\n### {scheme} -> {verdict}")
    for r in reasons:
        print(f"  - {r}")

verdict_df = pd.DataFrame(verdict_rows)
print()
print(verdict_df.to_string(index=False))

print()
print("구조적 위험 별도 명시(D가 E보다 OOF가 좋더라도 적용):")
print(f"  D(Notion 단순곱)는 클래스별 총weight 불균형(std={d_class_std:.4f})을 구조적으로 안고 있다.")
print(f"  이 불균형은 클래스 분포가 바뀌거나(파생변수 추가 등) 그룹 크기 분포가 달라지면")
print(f"  예측 불가능한 방향으로 더 커질 수 있다 — D의 OOF가 E보다 좋게 나오더라도, 그 이유가")
print(f"  '특정 클래스에 우연히 유리하게 쏠린 가중치' 때문일 위험이 있으므로 D를 그대로")
print(f"  채택하는 것은 권장하지 않는다(교정식 E를 우선 검토 대상으로 삼는다).")


### C_balanced_profile -> 기각(개선폭 부족)
  - B 대비 개선폭 -0.0017 (미충족 — 0.005 미만)
  - fold 개선 2/5 (미충족)
  - singleton-only delta +0.0185 (충족(악화 아님))
  - fold 표준편차 0.0180 (B의 1.22배, 충족)
  - 클래스 붕괴 점검: 0.05 넘게 악화된 클래스 2개, 0.01 넘게 개선된 클래스 11개 (정상 범위)

### D_balanced_same_label_naive -> 기각(개선폭 부족)
  - B 대비 개선폭 -0.0037 (미충족 — 0.005 미만)
  - fold 개선 1/5 (미충족)
  - singleton-only delta +0.0141 (충족(악화 아님))
  - fold 표준편차 0.0175 (B의 1.18배, 충족)
  - 클래스 붕괴 점검: 0.05 넘게 악화된 클래스 3개, 0.01 넘게 개선된 클래스 10개 (정상 범위)

### E_same_label_rebalanced -> 기각(개선폭 부족)
  - B 대비 개선폭 -0.0028 (미충족 — 0.005 미만)
  - fold 개선 1/5 (미충족)
  - singleton-only delta +0.0159 (충족(악화 아님))
  - fold 표준편차 0.0173 (B의 1.16배, 충족)
  - 클래스 붕괴 점검: 0.05 넘게 악화된 클래스 2개, 0.01 넘게 개선된 클래스 11개 (정상 범위)

                     scheme    verdict
         C_balanced_profile 기각(개선폭 부족)
D_balanced_same_label_naive 기각(개선폭 부족)
    E_same_label_rebalanced 기각(개선폭 부족)

구조적 위험 별도 명시(D가 E보다 OOF가 좋더라도 적용):
  D(Notion 단순곱)는 클래스별 총weight 불균형(std=11.6989)을 구조적으로 안고 있다.
  이 불균

## 산출물 저장

**이 셀이 하는 일**: 이 실험의 모든 산출물(스킴별 OOF CSV·metrics JSON, fold별 결과 CSV,
클래스별 F1 CSV, weight diagnostics CSV, provenance JSON)을 `artifacts/` 아래 새 파일로
저장한다. 기존 파일(06의 `xgb_enc3_sample27_skf5_s42` 계열, LR/RF 계열)과 이름이 겹치는지
먼저 검사하고, 겹치면 저장하지 않고 예외를 던진다.
**왜 필요한지**: 사용자가 나중에 이 노트북을 다시 열지 않고도 결과를 확인·재검증할 수
있어야 한다.
**입력**: 6~16절에서 계산된 모든 DataFrame/dict.
**출력**: `artifacts/oof/oof__xgboost__{run_id}.csv` × 5, `artifacts/metrics/metrics__xgboost__{run_id}.json` × 5,
`artifacts/metrics/fold_results__xgb_enc3_sample27_duplicate_weight_s42.csv`,
`artifacts/metrics/per_class_f1__xgb_enc3_sample27_duplicate_weight_s42.csv`,
`artifacts/metrics/weight_diagnostics__xgb_enc3_sample27_duplicate_weight_s42.csv`,
`artifacts/metrics/weight_per_class_total__xgb_enc3_sample27_duplicate_weight_s42.csv`,
`artifacts/metrics/provenance__xgb_enc3_sample27_duplicate_weight_s42.json`.
**누수 방지 조건**: 해당 없음(저장만).
**확인해야 할 부분**: 어떤 기존 파일도 덮어쓰지 않았는지(`_existing` 리스트가 비어 있는지).

In [12]:
import platform
from importlib.metadata import version as pkg_version

RUN_ID_BY_SCHEME = {spec["id"]: spec["run_id"] for spec in cfg["weight_schemes"]}

OOF_DIR = ARTIFACT_ROOT / "oof"
METRICS_DIR = ARTIFACT_ROOT / "metrics"
for d in (OOF_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

oof_paths = {scheme: OOF_DIR / f"oof__xgboost__{RUN_ID_BY_SCHEME[scheme]}.csv" for scheme in SCHEMES}
metrics_paths = {scheme: METRICS_DIR / f"metrics__xgboost__{RUN_ID_BY_SCHEME[scheme]}.json" for scheme in SCHEMES}
fold_results_path = METRICS_DIR / "fold_results__xgb_enc3_sample27_duplicate_weight_s42.csv"
per_class_f1_path = METRICS_DIR / "per_class_f1__xgb_enc3_sample27_duplicate_weight_s42.csv"
weight_diag_path = METRICS_DIR / "weight_diagnostics__xgb_enc3_sample27_duplicate_weight_s42.csv"
weight_class_total_path = METRICS_DIR / "weight_per_class_total__xgb_enc3_sample27_duplicate_weight_s42.csv"
provenance_out_path = METRICS_DIR / "provenance__xgb_enc3_sample27_duplicate_weight_s42.json"

_existing = [p for p in (list(oof_paths.values()) + list(metrics_paths.values())
                         + [fold_results_path, per_class_f1_path, weight_diag_path,
                            weight_class_total_path, provenance_out_path]) if p.exists()]
if _existing:
    raise RuntimeError(f"이미 있는 산출물 경로 — 덮어쓰지 않는다: {_existing}")


def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# 1) 스킴별 OOF CSV
for scheme in SCHEMES:
    oof_df = pd.DataFrame(OOF_PROBA[scheme], columns=PROBA_COLUMNS)
    oof_df.insert(0, "ID", train_df["ID"].to_numpy())
    oof_df.insert(1, "SUBCLASS", train_df["SUBCLASS"].to_numpy())
    oof_df.insert(2, "fold", fold_assignment)
    oof_df["prediction"] = oof_pred[scheme]
    oof_df.to_csv(oof_paths[scheme], index=False)

# 2) fold별 결과 CSV, 클래스별 F1 CSV, weight 진단 CSV
fold_metrics_df.to_csv(fold_results_path, index=False)
per_class_f1_df.to_csv(per_class_f1_path)
weight_diagnostics_df.to_csv(weight_diag_path, index=False)
per_class_total_df.to_csv(weight_class_total_path, index=False)

# 3) 스킴별 metrics JSON
package_versions = {}
for pkg in ("numpy", "pandas", "scikit-learn", "xgboost", "scipy", "pyarrow", "pyyaml"):
    try:
        package_versions[pkg] = pkg_version(pkg)
    except Exception:
        package_versions[pkg] = "not installed"

executed_at = datetime.now(timezone.utc).isoformat()
for scheme in SCHEMES:
    scheme_fold_rows = [r for r in fold_metrics_rows if r["scheme"] == scheme]
    per_class_report = {
        cls: float(per_class_f1_df.loc[cls, scheme]) for cls in per_class_f1_df.index
    }
    verdict_row = verdict_df.loc[verdict_df["scheme"] == scheme, "verdict"]
    verdict_value = verdict_row.iloc[0] if len(verdict_row) else "기준선(A/B) — 판정 대상 아님"
    record = {
        "run_id": RUN_ID_BY_SCHEME[scheme],
        "executed_at_utc": executed_at,
        "model_name": "xgboost",
        "scheme": scheme,
        "config": {"path": str(CONFIG_PATH), "sha256": sha256_of_file(CONFIG_PATH)},
        "train_feature": {"path": str(TRAIN_FEATURE_PATH), "sha256": sha256_of_file(TRAIN_FEATURE_PATH)},
        "feature_manifest": {"path": str(MANIFEST_PATH), "sha256": sha256_of_file(MANIFEST_PATH)},
        "group5_fold_assignment": {"path": str(GROUP5_FOLD_PATH), "sha256": sha256_of_file(GROUP5_FOLD_PATH)},
        "group5_fold_provenance": group5_provenance,
        "feature_count": len(GENE_ENCODING_COLUMNS) + len(OTHER_FEATURE_COLUMNS) + len(RATIO_TRANSFORM_FEATURE_COLUMNS),
        "class_order": CLASS_ORDER,
        "fold": {"strategy": "StratifiedGroupKFold via make_profile_group_kfold(develop)", "n_splits": N_SPLITS, "numbering": "0-based"},
        "model_params": XGB_PARAMS,
        "early_stopping_used": False,
        "fold_macro_f1": scheme_fold_rows,
        "fold_macro_f1_mean": float(fold_mean_std_df.loc[scheme, "mean"]),
        "fold_macro_f1_std": float(fold_mean_std_df.loc[scheme, "std"]),
        "oof_macro_f1": float(oof_macro_f1[scheme]),
        "oof_macro_f1_delta_vs_A": float(oof_summary_df.set_index("scheme").loc[scheme, "delta_vs_A"]),
        "oof_macro_f1_delta_vs_B": float(oof_summary_df.set_index("scheme").loc[scheme, "delta_vs_B"]),
        "per_class_f1": per_class_report,
        "singleton_only_macro_f1": float(singleton_vs_duplicate_df.set_index("scheme").loc[scheme, "singleton_macro_f1"]),
        "duplicate_only_macro_f1": float(singleton_vs_duplicate_df.set_index("scheme").loc[scheme, "duplicate_macro_f1"]),
        "n_singleton": int(singleton_mask.sum()),
        "n_duplicate": int(duplicate_mask.sum()),
        "disagreement_rate_vs_A": None if scheme == "A_none" else float(disagreement_df.set_index("scheme").loc[scheme, "disagreement_rate_vs_A"]),
        "weight_diagnostics_foldmean": (
            weight_diagnostics_df.loc[weight_diagnostics_df["scheme"] == scheme,
                ["norm_mean", "norm_std", "norm_min", "norm_max", "effective_sample_size", "ess_ratio"]]
            .mean().to_dict() if scheme != "A_none" else None
        ),
        "verdict": verdict_value,
        "package_versions": package_versions,
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "leakage_controls": [
            "RatioTransformFeatures를 fold마다 새로 만들어 train fold에만 fit, val은 transform만",
            "sample_weight는 fold-local WEIGHT_CACHE에서만 가져옴 — 전체 train에서 미리 계산한 적 없음",
            "Group5 fold는 원본 train.csv의 4,384개 변이 문자열 열로 만든 profile_hash 기반, enc3 인코딩값 아님",
            "test 데이터는 이 실험에서 아예 로드하지 않음 — 제출 예측 생성 안 함",
            "eval_set/early stopping 미사용",
        ],
        "note": "실제 DACON 제출은 수행하지 않았다 — 이 실험은 OOF 비교 전용이며 test 추론/제출 후보를 만들지 않는다.",
    }
    with open(metrics_paths[scheme], "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

# 4) 종합 provenance JSON
provenance_out = {
    "created_at_utc": executed_at,
    "experiment": "08_duplicate_group_weight_experiment",
    "onco_ai_git_head": _head,
    "onco_ai_git_dirty": _dirty,
    "raw_train_csv_sha256_via_group5_provenance": group5_provenance["raw_train_csv"],
    "train_feature_parquet_sha256": sha256_of_file(TRAIN_FEATURE_PATH),
    "feature_manifest_sha256": sha256_of_file(MANIFEST_PATH),
    "group5_fold_sha256": sha256_of_file(GROUP5_FOLD_PATH),
    "group5_fold_provenance": group5_provenance,
    "seed": 42,
    "n_splits": N_SPLITS,
    "schemes": SCHEMES,
    "xgb_params": XGB_PARAMS,
    "oof_summary": oof_summary_df.to_dict(orient="records"),
    "verdicts": verdict_df.to_dict(orient="records"),
    "package_versions": package_versions,
    "not_submitted_to_dacon": True,
    "team_repo_modified": False,
}
with open(provenance_out_path, "w", encoding="utf-8") as f:
    json.dump(provenance_out, f, indent=2, ensure_ascii=False)

print("저장된 파일:")
for p in list(oof_paths.values()) + list(metrics_paths.values()) + [
    fold_results_path, per_class_f1_path, weight_diag_path, weight_class_total_path, provenance_out_path
]:
    print(f"  {p}")


저장된 파일:
  <개인 workspace 경로>/artifacts/oof/oof__xgboost__xgb_enc3_sample27_group5_w_none_s42.csv
  <개인 workspace 경로>/artifacts/oof/oof__xgboost__xgb_enc3_sample27_group5_w_balanced_s42.csv
  <개인 workspace 경로>/artifacts/oof/oof__xgboost__xgb_enc3_sample27_group5_w_bal_profile_s42.csv
  <개인 workspace 경로>/artifacts/oof/oof__xgboost__xgb_enc3_sample27_group5_w_bal_same_label_naive_s42.csv
  <개인 workspace 경로>/artifacts/oof/oof__xgboost__xgb_enc3_sample27_group5_w_same_label_rebalanced_s42.csv
  <개인 workspace 경로>/artifacts/metrics/metrics__xgboost__xgb_enc3_sample27_group5_w_none_s42.json
  <개인 workspace 경로>/artifacts/metrics/metrics__xgboost__xgb_enc3_sample27_group5_w_balanced_s42.json
  <개인 workspace 경로>/artifacts/metrics/metrics__xgboost__xgb_enc3_sample27_group5_w_bal_profile_s42.json
  <개인 workspace 경로>/artifacts/metrics/metrics__xgboost__xgb_enc3_sample27_group5_w_bal_same_label_naive_s42.json
  <개인 workspace 경로>/artifacts/metrics/metrics__xgboost__xgb_enc3_sample27_group5_w_same_label

## 17. 다음 단계

이 노트북은 **비교 실험**이며, 어떤 스킴도 이 노트북 실행만으로 자동 채택되지 않는다.
16절의 판정은 사전에 고정한 기준을 코드로 적용한 결과일 뿐, 최종 채택 여부는 사용자
승인을 거쳐야 한다.

다음에 고려할 것(이번 실험 범위 밖 — 실행하지 않음):
1. 채택 후보로 판정된 스킴이 있다면, 06_xgb_enc3_sample27.ipynb의 정식 산출물(OOF/metrics)
   경로에 **정식 실험으로 승격**할지 사용자와 논의한다(현재는 이 노트북 전용 경로에만
   저장했다 — 06의 산출물을 덮어쓰지 않았다).
2. `power` 파라미터(그룹 크기 감쇠 강도)를 1.0 고정이 아니라 탐색해볼 여지가 있다 —
   단, 이번 실험 범위가 아니므로 여기서는 손대지 않았다.
3. Group5 CV 자체(가중치와 무관하게)가 기존 SKF 대비 OOF에 미치는 순수 효과는 이
   노트북만으로는 분리되지 않는다(모든 스킴이 Group5를 공통으로 쓰기 때문) — 별도
   비교가 필요하면 후속 실험으로 제안한다.
4. 딥러닝, Optuna, SHAP, 새 파생변수, 제출 후보 생성은 이번 범위가 아니다.
5. DACON 제출은 사용자 승인 없이는 하지 않는다.